# 04 — Bottleneck Analysis
Measure waiting time and service time at every activity step.
Identify which steps, roles, and departments cause the most delay.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.load_event_log import load_xes_log, convert_to_dataframe

DATA_PATH   = Path('../data/raw/PermitLog.xes')
FIGURES_OUT = Path('../outputs/figures')
TABLES_OUT  = Path('../outputs/tables')
FIGURES_OUT.mkdir(parents=True, exist_ok=True)
TABLES_OUT.mkdir(parents=True, exist_ok=True)

In [2]:
log = load_xes_log(DATA_PATH)
df  = convert_to_dataframe(log)
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], utc=True)
df = df.sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)
print(f'{len(df):,} events | {df["case:concept:name"].nunique():,} cases')

/opt/anaconda3/lib/python3.12/site-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/7065 [00:00<?, ?it/s]

86,581 events | 7,065 cases


## 1. Compute waiting time and service time for every event

- **Waiting time** = gap between the *previous* event and this event (inbound delay)
- **Service time** = gap between this event and the *next* event in the same case (how long before the process moves on)

In [3]:
grp = df.groupby('case:concept:name')['time:timestamp']

df['prev_ts']       = grp.shift(1)
df['next_ts']       = grp.shift(-1)
df['waiting_h']     = (df['time:timestamp'] - df['prev_ts']).dt.total_seconds() / 3600
df['service_h']     = (df['next_ts'] - df['time:timestamp']).dt.total_seconds() / 3600

# First event of each case has no waiting time; last has no service time — leave as NaN
print(f'Rows with waiting time: {df["waiting_h"].notna().sum():,}')
print(f'Rows with service time: {df["service_h"].notna().sum():,}')

Rows with waiting time: 79,516
Rows with service time: 79,516


## 2. Bottleneck table — waiting time per activity

In [4]:
wait_stats = (
    df.groupby('concept:name')['waiting_h']
    .agg(
        count='count',
        mean_wait_h='mean',
        median_wait_h='median',
        p75_wait_h=lambda x: x.quantile(0.75),
        p95_wait_h=lambda x: x.quantile(0.95),
        max_wait_h='max',
    )
    .sort_values('median_wait_h', ascending=False)
    .reset_index()
)
wait_stats[['mean_wait_h','median_wait_h','p75_wait_h','p95_wait_h','max_wait_h']] = \
    wait_stats[['mean_wait_h','median_wait_h','p75_wait_h','p95_wait_h','max_wait_h']].round(1)

wait_stats.to_csv(TABLES_OUT / 'bottleneck_waiting_time.csv', index=False)
print('Top 15 activities by median waiting time (hours):')
wait_stats.head(15)

Top 15 activities by median waiting time (hours):


,concept:name,count,mean_wait_h,median_wait_h,p75_wait_h,p95_wait_h,max_wait_h
0,Send Reminder,2303,1340.9,1266.7,1464.0,3372.8,9687.2
1,Permit SAVED by EMPLOYEE,5,656.6,508.6,880.1,1326.9,1438.6
2,Start trip,6331,776.1,465.4,1115.3,2482.2,8916.3
3,Permit REJECTED by MISSING,83,1008.9,353.9,1994.1,2890.2,4665.8
4,Permit REJECTED by DIRECTOR,2,300.7,300.7,391.3,463.7,481.8
5,Permit FOR_APPROVAL by SUPERVISOR,1,163.3,163.3,163.3,163.3,163.3
6,Declaration REJECTED by DIRECTOR,4,115.1,117.4,150.2,168.3,172.8
7,Declaration SAVED by EMPLOYEE,65,289.9,98.9,345.4,1029.4,1637.9
8,Declaration SUBMITTED by EMPLOYEE,7574,231.4,96.8,266.1,844.8,7100.4
9,End trip,7065,148.0,96.0,144.0,336.0,26448.0


## 3. Waiting time per activity — bar chart

In [5]:
# Top 20 activities by median wait, converted to days
top20 = wait_stats.head(20).copy()
top20['median_wait_d'] = top20['median_wait_h'] / 24
top20['p75_wait_d']    = top20['p75_wait_h']    / 24

fig, ax = plt.subplots(figsize=(13, 8))
bars = ax.barh(top20['concept:name'], top20['median_wait_d'], color='#2c7bb6', label='Median wait (days)')
ax.barh(top20['concept:name'], top20['p75_wait_d'] - top20['median_wait_d'],
        left=top20['median_wait_d'], color='#abd9e9', alpha=0.7, label='75th percentile')

for bar, val in zip(bars, top20['median_wait_d']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}d', va='center', fontsize=8)

ax.set_xlabel('Waiting time (days)')
ax.set_title('Median Waiting Time per Activity (Top 20)', fontsize=13)
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'bottleneck_waiting_time.png', dpi=150)
plt.show()
print('Saved bottleneck_waiting_time.png')

Saved bottleneck_waiting_time.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/3015698251.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Service time per activity

In [6]:
svc_stats = (
    df.groupby('concept:name')['service_h']
    .agg(
        count='count',
        mean_svc_h='mean',
        median_svc_h='median',
        p75_svc_h=lambda x: x.quantile(0.75),
        p95_svc_h=lambda x: x.quantile(0.95),
    )
    .sort_values('median_svc_h', ascending=False)
    .reset_index()
)
svc_stats[['mean_svc_h','median_svc_h','p75_svc_h','p95_svc_h']] = \
    svc_stats[['mean_svc_h','median_svc_h','p75_svc_h','p95_svc_h']].round(1)

svc_stats.to_csv(TABLES_OUT / 'bottleneck_service_time.csv', index=False)
print('Top 15 activities by median service time (hours):')
svc_stats.head(15)

Top 15 activities by median service time (hours):


,concept:name,count,mean_svc_h,median_svc_h,p75_svc_h,p95_svc_h
0,Send Reminder,1312,929.3,1416.0,1464.0,1488.0
1,Permit SAVED by EMPLOYEE,13,1826.1,1016.8,1785.7,5570.1
2,Permit FINAL_APPROVED by DIRECTOR,681,807.0,468.1,1130.7,2486.7
3,Payment Handled,2070,718.1,438.5,1038.5,2299.7
4,Permit FINAL_APPROVED by SUPERVISOR,6286,650.5,325.4,903.2,2321.9
5,Permit FOR_APPROVAL by ADMINISTRATION,1,276.5,276.5,276.5,276.5
6,Permit REJECTED by DIRECTOR,2,240.7,240.7,300.9,349.0
7,End trip,6612,522.4,178.5,689.5,1465.8
8,Declaration REJECTED by DIRECTOR,4,375.7,159.2,529.2,1049.9
9,Permit FOR_APPROVAL by SUPERVISOR,1,120.7,120.7,120.7,120.7


## 5. Combined bottleneck view — waiting + service time

In [7]:
combined = wait_stats[['concept:name','count','median_wait_h']].merge(
    svc_stats[['concept:name','median_svc_h']], on='concept:name'
)
combined['total_delay_h']  = combined['median_wait_h'] + combined['median_svc_h']
combined['total_delay_d']  = (combined['total_delay_h'] / 24).round(1)
combined['median_wait_d']  = (combined['median_wait_h']  / 24).round(1)
combined['median_svc_d']   = (combined['median_svc_h']   / 24).round(1)
combined = combined.sort_values('total_delay_d', ascending=False).reset_index(drop=True)
combined.to_csv(TABLES_OUT / 'bottleneck_combined.csv', index=False)

top15c = combined.head(15)
fig, ax = plt.subplots(figsize=(13, 8))
ax.barh(top15c['concept:name'], top15c['median_wait_d'],  color='#d7191c', label='Median wait (days)')
ax.barh(top15c['concept:name'], top15c['median_svc_d'],
        left=top15c['median_wait_d'], color='#fdae61', label='Median service (days)')
ax.set_xlabel('Days')
ax.set_title('Total Delay per Activity — Wait + Service Time (Top 15)', fontsize=12)
ax.invert_yaxis()
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'bottleneck_combined.png', dpi=150)
plt.show()
print('Saved bottleneck_combined.png')

Saved bottleneck_combined.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/1040986369.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Delay by organisational entity (department)

In [8]:
dept_stats = (
    df.groupby('case:OrganizationalEntity')['waiting_h']
    .agg(n_events='count', median_wait_h='median', mean_wait_h='mean')
    .sort_values('median_wait_h', ascending=False)
    .reset_index()
)
dept_stats['median_wait_d'] = (dept_stats['median_wait_h'] / 24).round(1)
dept_stats['mean_wait_d']   = (dept_stats['mean_wait_h']   / 24).round(1)
dept_stats.to_csv(TABLES_OUT / 'bottleneck_by_department.csv', index=False)

top_dept = dept_stats.head(20)
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_dept['case:OrganizationalEntity'].astype(str), top_dept['median_wait_d'], color='#2c7bb6')
ax.set_xlabel('Median waiting time (days)')
ax.set_title('Median Waiting Time by Department (Top 20)', fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'bottleneck_by_department.png', dpi=150)
plt.show()
print('Saved bottleneck_by_department.png')

Saved bottleneck_by_department.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/3678031167.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Delay by resource role

In [9]:
role_col = 'org:role' if 'org:role' in df.columns else None
if role_col:
    role_stats = (
        df.groupby(role_col)['waiting_h']
        .agg(n_events='count', median_wait_h='median', mean_wait_h='mean')
        .sort_values('median_wait_h', ascending=False)
        .reset_index()
    )
    role_stats['median_wait_d'] = (role_stats['median_wait_h'] / 24).round(1)
    role_stats.to_csv(TABLES_OUT / 'bottleneck_by_role.csv', index=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(role_stats[role_col], role_stats['median_wait_d'], color='#1a9641')
    ax.set_xlabel('Median waiting time (days)')
    ax.set_title('Median Waiting Time by Role', fontsize=12)
    ax.invert_yaxis()
    plt.tight_layout()
    fig.savefig(FIGURES_OUT / 'bottleneck_by_role.png', dpi=150)
    plt.show()
    print(role_stats.to_string(index=False))
else:
    print('No org:role column found')

      org:role  n_events  median_wait_h  mean_wait_h  median_wait_d
      EMPLOYEE     26371     110.042500   342.747195            4.6
       MISSING       186      97.062639   513.008281            4.0
     UNDEFINED     17388      72.479722   242.466329            3.0
      DIRECTOR       941      24.884167    75.145352            1.0
    SUPERVISOR     14896      22.813889    54.044429            1.0
  BUDGET OWNER      4469      22.142222    54.752730            0.9
ADMINISTRATION     13837       0.003889    23.247229            0.0
  PRE_APPROVER      1428       0.003889    18.032887            0.0


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/4021158281.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Stuck cases — ended on Send Reminder

In [10]:
last_event = (
    df.sort_values(['case:concept:name','time:timestamp'])
    .groupby('case:concept:name')
    .last()
    .reset_index()[['case:concept:name','concept:name','time:timestamp',
                     'case:OrganizationalEntity','case:TotalDeclared']]
)
stuck = last_event[last_event['concept:name'] == 'Send Reminder'].copy()
print(f'Stuck cases (last event = Send Reminder): {len(stuck)}')
stuck.to_csv(TABLES_OUT / 'stuck_cases.csv', index=False)

Stuck cases (last event = Send Reminder): 991


In [11]:
# Duration of stuck cases vs resolved cases
from src.process_summary import get_case_durations
durations = get_case_durations(df)
durations['stuck'] = durations['case:concept:name'].isin(stuck['case:concept:name'])

summary = durations.groupby('stuck')['duration_days'].describe()[['count','mean','50%','75%','max']].round(1)
summary.index = ['Resolved', 'Stuck (Send Reminder)']
print(summary.to_string())
summary.to_csv(TABLES_OUT / 'stuck_vs_resolved_duration.csv')

                        count   mean    50%    75%     max
Resolved               6074.0   76.2   63.2   98.4  1190.3
Stuck (Send Reminder)   991.0  156.3  133.8  175.7   495.9


In [12]:
fig, ax = plt.subplots(figsize=(10, 5))
resolved_d = durations[~durations['stuck']]['duration_days'].clip(upper=400)
stuck_d    = durations[ durations['stuck']]['duration_days'].clip(upper=400)

ax.hist(resolved_d, bins=50, alpha=0.6, color='#2c7bb6', label=f'Resolved ({(~durations["stuck"]).sum():,} cases)')
ax.hist(stuck_d,    bins=30, alpha=0.7, color='#d7191c', label=f'Stuck — Send Reminder ({durations["stuck"].sum():,} cases)')
ax.set_xlabel('Case duration (days, clipped at 400)')
ax.set_ylabel('Number of cases')
ax.set_title('Case Duration: Resolved vs Stuck Cases')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'stuck_vs_resolved_duration.png', dpi=150)
plt.show()
print('Saved stuck_vs_resolved_duration.png')

Saved stuck_vs_resolved_duration.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/1411287329.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# Which departments have the most stuck cases?
stuck_dept = (
    stuck.groupby('case:OrganizationalEntity').size()
    .reset_index(name='stuck_count')
    .sort_values('stuck_count', ascending=False)
)
stuck_dept.to_csv(TABLES_OUT / 'stuck_by_department.csv', index=False)

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(stuck_dept['case:OrganizationalEntity'].astype(str), stuck_dept['stuck_count'], color='#d7191c')
ax.set_xlabel('Number of stuck cases')
ax.set_title('Stuck Cases (Send Reminder) by Department')
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'stuck_by_department.png', dpi=150)
plt.show()
print(stuck_dept.to_string(index=False))

case:OrganizationalEntity  stuck_count
organizational unit 65458          216
organizational unit 65455          172
organizational unit 65454          107
organizational unit 65456           98
organizational unit 65459           95
organizational unit 65466           61
organizational unit 65457           58
organizational unit 65460           56
organizational unit 65464           53
organizational unit 65477           18
organizational unit 65461           17
organizational unit 65469           11
organizational unit 65470            6
organizational unit 65472            5
organizational unit 65475            5
organizational unit 65467            4
organizational unit 65474            3
organizational unit 65465            3
organizational unit 65473            1
organizational unit 65462            1
organizational unit 65482            1


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/3348772466.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Monthly event volume — process load over time

In [14]:
df['month'] = df['time:timestamp'].dt.to_period('M')
monthly = df.groupby('month').size().reset_index(name='event_count')
monthly['month_str'] = monthly['month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(monthly['month_str'], monthly['event_count'], color='#2c7bb6', width=0.8)
ax.set_xlabel('Month')
ax.set_ylabel('Number of events')
ax.set_title('Monthly Event Volume')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'monthly_event_volume.png', dpi=150)
plt.show()
print('Saved monthly_event_volume.png')

/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/2717009584.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['month'] = df['time:timestamp'].dt.to_period('M')


Saved monthly_event_volume.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/2717009584.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Summary — key bottleneck findings

In [15]:
print('=== TOP 10 BOTTLENECKS (by total delay = wait + service, days) ===')
print(combined[['concept:name','median_wait_d','median_svc_d','total_delay_d','count']].head(10).to_string(index=False))
print()
print('=== STUCK CASES SUMMARY ===')
print(f'  Total stuck: {len(stuck)} cases ({len(stuck)/7065*100:.1f}% of all cases)')
print(f'  Most affected dept: {stuck_dept.iloc[0]["case:OrganizationalEntity"]} ({stuck_dept.iloc[0]["stuck_count"]} cases)')
print()
print('=== DURATION: STUCK vs RESOLVED ===')
print(summary.to_string())

=== TOP 10 BOTTLENECKS (by total delay = wait + service, days) ===
                         concept:name  median_wait_d  median_svc_d  total_delay_d  count
                        Send Reminder           52.8          59.0          111.8   2303
             Permit SAVED by EMPLOYEE           21.2          42.4           63.6      5
                           Start trip           19.4           4.0           23.4   6331
          Permit REJECTED by DIRECTOR           12.5          10.0           22.6      2
                      Payment Handled            3.2          18.3           21.4   7544
    Permit FINAL_APPROVED by DIRECTOR            1.0          19.5           20.5    681
           Permit REJECTED by MISSING           14.7           3.3           18.0     83
  Permit FINAL_APPROVED by SUPERVISOR            0.9          13.6           14.5   6300
    Permit FOR_APPROVAL by SUPERVISOR            6.8           5.0           11.8      1
Permit FOR_APPROVAL by ADMINISTRATION      

## 11. Employee waiting: scheduling intervals vs administrative intervals

The critical review identified that the EMPLOYEE 4.6-day median aggregates two
categorically different waiting types:
- **Scheduling intervals** (`Start trip`, `End trip`): voluntary timing, not process delay
- **Administrative intervals** (all other employee activities): submission and resubmission behaviour

These must be separated before attributing delay to employee process behaviour.

In [16]:
SCHEDULING_ACTS = {'Start trip', 'End trip'}

emp = df[df['org:role'] == 'EMPLOYEE'].dropna(subset=['waiting_h']).copy()
emp['interval_type'] = emp['concept:name'].apply(
    lambda a: 'Scheduling' if a in SCHEDULING_ACTS else 'Administrative'
)
emp['waiting_d'] = emp['waiting_h'] / 24

split = (
    emp.groupby('interval_type')['waiting_d']
    .agg(n='count', median='median', mean='mean',
         p25=lambda x: x.quantile(0.25), p75=lambda x: x.quantile(0.75),
         p95=lambda x: x.quantile(0.95))
    .round(2)
)
print("Employee waiting split by interval type (days):")
print(split.to_string())
split.to_csv(TABLES_OUT / 'employee_wait_split.csv')


Employee waiting split by interval type (days):
                    n  median   mean   p25    p75    p95
interval_type                                           
Administrative  12975    3.47   9.89  0.78  10.19  38.87
Scheduling      13396    5.56  18.54  2.30  19.64  80.55


In [17]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Scheduling': '#2c7bb6', 'Administrative': '#d7191c'}

for ax, (itype, grp) in zip(axes, emp.groupby('interval_type')):
    clipped = grp['waiting_d'].clip(upper=grp['waiting_d'].quantile(0.99))
    ax.hist(clipped, bins=40, color=colors[itype], edgecolor='white', alpha=0.85)
    med = grp['waiting_d'].median()
    ax.axvline(med, color='black', linestyle='--', linewidth=1.5, label=f'Median = {med:.1f}d')
    ax.set_title(f'EMPLOYEE — {itype} intervals\n(n={len(grp):,})', fontsize=11)
    ax.set_xlabel('Waiting time (days, 99th pct clip)')
    ax.set_ylabel('Number of events')
    ax.legend(fontsize=9)

plt.suptitle('Employee Waiting Time: Scheduling vs Administrative Intervals', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'employee_wait_split.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved employee_wait_split.png')


Saved employee_wait_split.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/708836366.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Contribution of each interval type to total case duration
from src.process_summary import get_case_durations
dur = get_case_durations(df)
dur_map = dur.set_index('case:concept:name')['duration_hours']

for itype in ['Scheduling', 'Administrative']:
    grp_sum = emp[emp['interval_type'] == itype].groupby('case:concept:name')['waiting_h'].sum()
    cases   = grp_sum.index.intersection(dur_map.index)
    pct     = (grp_sum[cases] / dur_map[cases] * 100).dropna()
    print(f"{itype} intervals — median share of case duration: {pct.median():.1f}%  "
          f"(mean {pct.mean():.1f}%, n={len(pct)})")


Scheduling intervals — median share of case duration: 35.3%  (mean 39.5%, n=7065)
Administrative intervals — median share of case duration: 20.2%  (mean 26.9%, n=5980)


## 12. ADMINISTRATION waiting — full distribution and tail

The 0-day median is accurate but the distribution is right-skewed with a non-negligible tail.
A log-scale view and CDF reveal what the median conceals.

In [19]:
adm = df[df['org:role'] == 'ADMINISTRATION'].dropna(subset=['waiting_h']).copy()
adm['waiting_d'] = adm['waiting_h'] / 24

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: log-scale histogram
ax = axes[0]
nonzero = adm[adm['waiting_d'] > 0]['waiting_d']
ax.hist(nonzero.clip(upper=nonzero.quantile(0.999)), bins=50,
        color='#2c7bb6', edgecolor='white', alpha=0.85)
ax.set_xscale('log')
ax.set_xlabel('Waiting time (days, log scale, non-zero only)')
ax.set_ylabel('Number of events')
ax.set_title(f'ADMINISTRATION — Non-zero waiting times\n'
             f'({len(nonzero):,} of {len(adm):,} events, {len(nonzero)/len(adm)*100:.1f}%)',
             fontsize=10)
ax.text(0.98, 0.95, f'p95 = {adm["waiting_d"].quantile(0.95):.1f}d\n'
                    f'p99 = {adm["waiting_d"].quantile(0.99):.1f}d\n'
                    f'max = {adm["waiting_d"].max():.1f}d',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Right: ECDF
ax2 = axes[1]
sorted_d = adm['waiting_d'].sort_values().values
ecdf     = (1 + np.arange(len(sorted_d))) / len(sorted_d)
ax2.plot(sorted_d, ecdf * 100, color='#2c7bb6', linewidth=1.5)
for threshold, label, color in [(1, '1 day', '#1a9641'), (7, '7 days', '#fdae61'),
                                  (30, '30 days', '#d7191c')]:
    pct_w = (adm['waiting_d'] <= threshold).mean() * 100
    ax2.axvline(threshold, color=color, linestyle='--', alpha=0.7,
                label=f'≤{label}: {pct_w:.1f}%')
ax2.set_xlim(0, 60)
ax2.set_xlabel('Waiting time (days, clipped at 60)')
ax2.set_ylabel('Cumulative % of events')
ax2.set_title('ADMINISTRATION — Empirical CDF', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('ADMINISTRATION Waiting Time Distribution', fontsize=12)
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'admin_wait_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved admin_wait_distribution.png')


Saved admin_wait_distribution.png


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/3183712559.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Right-censoring — cases opened vs closed over time

The 2019 decline in event volume reflects dataset truncation, not operational wind-down.
No cases were opened after December 2018. All 2019 activity is tail-end resolution
of cases initiated in 2018. This chart distinguishes case opening from case closing.

In [20]:
case_first = df.groupby('case:concept:name')['time:timestamp'].min()
case_last  = df.groupby('case:concept:name')['time:timestamp'].max()

opened_m = case_first.dt.to_period('M').value_counts().sort_index()
closed_m = case_last.dt.to_period('M').value_counts().sort_index()

all_months = opened_m.index.union(closed_m.index)
opened_m   = opened_m.reindex(all_months, fill_value=0)
closed_m   = closed_m.reindex(all_months, fill_value=0)

x = [str(m) for m in all_months]
xi = range(len(x))

fig, ax = plt.subplots(figsize=(16, 5))
width = 0.4
ax.bar([i - width/2 for i in xi], opened_m.values, width=width,
       color='#2c7bb6', label='Cases opened')
ax.bar([i + width/2 for i in xi], closed_m.values, width=width,
       color='#d7191c', alpha=0.7, label='Cases closed')
ax.set_xticks(list(xi))
ax.set_xticklabels(x, rotation=45, ha='right', fontsize=7)
ax.set_ylabel('Number of cases')
ax.set_title('Cases Opened vs Closed per Month\n'
             '(No cases opened after Dec 2018 — 2019 activity is right-censored tail)', fontsize=11)
ax.axvline(x.index('2018-12') + 0.5, color='black', linestyle='--',
           linewidth=1.5, label='Last case opened (Dec 2018)')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_OUT / 'cases_opened_vs_closed.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved cases_opened_vs_closed.png')

n_right_censored = (case_last.dt.year >= 2019).sum()
print(f'\nCases opened in 2018 but closed 2019+: {n_right_censored}')
print(f'These represent right-censored observations for duration analysis.')


Saved cases_opened_vs_closed.png

Cases opened in 2018 but closed 2019+: 805
These represent right-censored observations for duration analysis.


/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/2374614738.py:4: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  opened_m = case_first.dt.to_period('M').value_counts().sort_index()
/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/2374614738.py:5: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  closed_m = case_last.dt.to_period('M').value_counts().sort_index()
/var/folders/_q/l3d3lyg51ms6dcm4x_hjt6vw0000gn/T/ipykernel_96059/2374614738.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
